In [19]:
import re
from pathlib import Path
import math
from collections import Counter
from typing import Dict, List, Sequence, Tuple

# Tokenization

---
**resources**:
- [Huggingface tokenizer summary](https://huggingface.co/docs/transformers/tokenizer_summary)
---

converts any string into a sequence of tokens.

### Split by spaces
- this doesn't work for some languages such as Chinese, where no spaces between words:
- some languages have long compound words, such as German, where splitting by spaces would break the meaning of the word.
- There are hyphenated words, such as "father-in-law", and contractions, like "don't", which should get split up.

- we don't want too **many** tokens, or else the sequences become difficult to model
- don't want too **few** tokens, or else there won't be parameter sharing between words.
- Each token should be a linguistically or statistically meaningful unit.

### Byte pair encoding

- start with a base vocabulary of characters, and then iteratively merge the most common pairs of tokens into new tokens until we reach a desired vocabulary size.

For example, give "the car", "the cat", "the rat":
- start with \[t, h, e, _, c, a, r],  \[t, h, e, _, c, a, t],  \[t, h, e, _, r, a, t]
- \[th, e, _, c, a, r],  \[th, e, _, c, a, t],  \[th, e, _, r, a, t] $\leftarrow$ merge "t" and "h", as they are the most common pair of tokens.
- \[the, _, c, a, r],  \[the, _, c, a, t],  \[the, _, r, a, t]
- \[the, _, ca, r],  \[the, _, ca, t],  \[the, _, r, a, t]

The output of the learning is:
- updated vocabulary $\mathcal{V}$: \{a, c, e, h, t, r, ca, th, the,\}
- the merges that we made:
    - th $\to$ t + h
    - the $\to$ th + e
    - ca $\to$ c + a

**Applying the tokenizer**: to tokenize a new string, apply the merges in the same order.

In [2]:
def clean_corpus_text(text: str) -> str:
    # Light normalization for tokenizer training data.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[\t\f\v ]+", " ", text)    # collapse horizontal whitespace
    text = re.sub(r" *\n *", "\n", text)       # trim spaces around line breaks
    text = re.sub(r"\n{3,}", "\n\n", text)     # collapse excessive blank lines
    return text.strip() + "\n"


In [4]:
raw_corpus_path = Path("../assets/tokenizer_training_corpus.txt")
clean_corpus_path = Path("../assets/tokenizer_training_corpus.cleaned.txt")

raw_text = raw_corpus_path.read_text(encoding="utf-8")
clean_text = clean_corpus_text(raw_text)
clean_corpus_path.write_text(clean_text, encoding="utf-8")

print(f"Raw chars: {len(raw_text):,}")
print(f"Clean chars: {len(clean_text):,}")
print(f"Saved cleaned corpus to: {clean_corpus_path.resolve()}")

Raw chars: 7,404
Clean chars: 7,402
Saved cleaned corpus to: /Users/haominfenh/Documents/Python/path-to-my-agent/assets/tokenizer_training_corpus.cleaned.txt


### BPE Implementation Notes

This BPE implementation follows the classic merge-based training loop.

1. Split text into words and represent each word as characters plus an end-of-word marker (`</w>`).
2. Count adjacent token-pair frequencies across the corpus (weighted by word frequency).
3. Merge the most frequent pair into a new token.
4. Repeat until target vocabulary size (or min pair frequency stop condition).

Function guide:
- `_get_pair_counts`: counts adjacent token pairs used to select the next merge.
- `_merge_pair_in_word`: applies one merge operation to one tokenized word.
- `train`: learns merges and vocabulary from corpus text.
- `encode_word` / `encode`: applies learned merges in learned order.
- `decode`: reconstructs text by removing end-of-word markers and joining words.


In [11]:
class BPE:
    def __init__(self):
        # `</w>` marks the word boundary so decode can reconstruct word spacing.
        self.end_of_word = "</w>"
        # Ordered list of merges learned during training.
        self.merges = []
        # Fast lookup: pair -> merge rank (lower rank = earlier merge).
        self.merge_ranks = {}
        # Token vocabulary (starts with chars, grows with merged subwords).
        self.vocab = set()

    @staticmethod
    def _get_pair_counts(tokenized_words, word_freq):
        """Count adjacent token-pair frequencies over the corpus."""
        pair_counts = Counter()
        for symbols, freq in word_freq.items():
            pieces = tokenized_words[symbols]
            for i in range(len(pieces) - 1):
                pair_counts[(pieces[i], pieces[i + 1])] += freq
        return pair_counts

    @staticmethod
    def _merge_pair_in_word(pieces, pair):
        """Merge occurrences of `pair` inside a tokenized word sequence."""
        i = 0
        merged = []
        while i < len(pieces):
            if i < len(pieces) - 1 and pieces[i] == pair[0] and pieces[i + 1] == pair[1]:
                merged.append(pair[0] + pair[1])
                i += 2
            else:
                merged.append(pieces[i])
                i += 1
        return tuple(merged)

    def train(self, text, vocab_size=300, min_pair_freq=2):
        """
        Learn BPE merges from text.

        @:param text: Preprocessed corpus text.
        @:param vocab_size: Target maximum vocabulary size.
        @:param min_pair_freq: Stop when the best pair frequency drops below this.
        """
        # Split corpus into space-separated words.
        words = re.findall(r"\S+", text)
        if not words:
            raise ValueError("Training text is empty after preprocessing.")

        # Represent each word as characters + end-of-word marker.
        # Example: "cat" -> ('c', 'a', 't', '</w>').
        word_freq = Counter(tuple(list(word) + [self.end_of_word]) for word in words)
        # A mapping from each word-symbol tuple to itself, this gets updated as merges happen
        tokenized_words = {symbols: symbols for symbols in word_freq}

        self.vocab = {self.end_of_word}
        for symbols in word_freq:
            self.vocab.update(symbols[:-1])

        self.merges = []
        self.merge_ranks = {}

        # Iteratively merge the most frequent adjacent token pair.
        while len(self.vocab) < vocab_size:
            pair_counts = self._get_pair_counts(tokenized_words, word_freq)
            if not pair_counts:
                break

            best_pair, best_freq = max(pair_counts.items(), key=lambda x: x[1])
            if best_freq < min_pair_freq:
                break

            self.merges.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merges) - 1
            # New merged token becomes part of vocabulary.
            self.vocab.add(best_pair[0] + best_pair[1])

            # Apply the chosen merge across all unique words.
            for symbols in list(tokenized_words.keys()):
                tokenized_words[symbols] = self._merge_pair_in_word(tokenized_words[symbols], best_pair)

        return self

    def encode_word(self, word):
        """Tokenize a single word by repeatedly applying learned merges."""
        pieces = tuple(list(word) + [self.end_of_word])
        while True:
            candidate_pairs = []
            # collect possible merge options
            for i in range(len(pieces) - 1):
                pair = (pieces[i], pieces[i + 1])
                if pair in self.merge_ranks:
                    candidate_pairs.append((self.merge_ranks[pair], pair))

            if not candidate_pairs:
                break

            # apply the best merge option
            _, best_pair = min(candidate_pairs, key=lambda x: x[0])
            pieces = self._merge_pair_in_word(pieces, best_pair)

        return list(pieces)

    def encode(self, text):
        """Tokenize full text (whitespace split + per-word BPE encoding)."""
        words = re.findall(r"\S+", text)
        tokens = []
        for word in words:
            tokens.extend(self.encode_word(word))
        return tokens

    def decode(self, tokens):
        """Reconstruct text from BPE tokens using `</w>` boundaries."""
        words = []
        current = ""
        for token in tokens:
            if token.endswith(self.end_of_word):
                current += token[:-len(self.end_of_word)]
                words.append(current)
                current = ""
            else:
                current += token
        if current:
            words.append(current)
        return " ".join(words)


In [13]:
bpe = BPE().train(clean_text, vocab_size=800, min_pair_freq=2)

print(f"Learned merges: {len(bpe.merges)}")
print(f"Final vocab size: {len(bpe.vocab)}")

Learned merges: 697
Final vocab size: 790


In [14]:
sample_text = "Tokenizers are useful for efficient modeling."
encoded = bpe.encode(sample_text)
decoded = bpe.decode(encoded)

print("\nSample text:", sample_text)
print("Encoded tokens:", encoded)
print("Decoded text:", decoded)


Sample text: Tokenizers are useful for efficient modeling.
Encoded tokens: ['Tokenizers</w>', 'are</w>', 'u', 'se', 'ful</w>', 'for</w>', 'eff', 'ic', 'i', 'ent</w>', 'model', 'ing.</w>']
Decoded text: Tokenizers are useful for efficient modeling.


### Unigram

1. Start with a large set of candidate subwords, and each candidate gets a probability score based on how often it appears in the training data.
2. Scores how well the current vocabulary tokenizes the training data at each step
3. For every token, measure how much removing the token would increase the overall loss. For example, if removing a token caused many words to be split into smaller pieces, then the loss would increase a lot, and that token would be important to keep in the vocabulary.
4. Unigram removes the tokens with the lowest loss increase, usually the bottom 10-20%. Base characters will always be remained, so any word can be tokenized.
5. Steps 2-4 repeated until the desired vocabulary size is reached.

In [16]:
Token = str
TokenSequence = List[Token]

class Unigram:
    """A compact Unigram tokenizer with Viterbi segmentation and pruning."""

    def __init__(self, max_token_length: int = 12, unk_token: str = "<unk>") -> None:
        # Longest candidate token to consider when generating seed vocabulary.
        self.max_token_length: int = max_token_length
        self.unk_token: str = unk_token
        self.end_of_word: str = "</w>"

        # Final token set and token probabilities learned by training.
        self.vocab: set[str] = set()
        self.prob: Dict[Token, float] = {}

        # Base characters are never pruned, so every word stays segmentable.
        self.base_chars: set[str] = set()

    @staticmethod
    def _extract_words(text: str) -> List[str]:
        """Whitespace-level word extraction used by training and encoding."""
        return re.findall(r"\S+", text)

    def _seed_vocab(self, words: Sequence[str]) -> Counter[str]:
        """Generate initial candidate tokens from all substrings up to max length."""
        candidates: Counter[str] = Counter()
        for word in words:
            n: int = len(word)
            for i in range(n):
                for j in range(i + 1, min(n, i + self.max_token_length) + 1):
                    candidates[word[i:j]] += 1
        return candidates

    def _normalize_probs(self, usage: Counter[str], epsilon: float = 1e-12) -> None:
        """Update token probabilities from usage counts with light smoothing."""
        total: float = float(sum(usage.values())) + epsilon * max(1, len(usage))
        self.prob = {tok: (cnt + epsilon) / total for tok, cnt in usage.items()}

    def _viterbi_word(self, word: str) -> Tuple[TokenSequence, float]:
        """Best segmentation (minimum negative log-probability) for one word."""
        n: int = len(word)
        best_score: List[float] = [float("inf")] * (n + 1)
        best_path: List[int] = [-1] * (n + 1)

        best_score[0] = 0.0

        for end in range(1, n + 1):
            start_min: int = max(0, end - self.max_token_length)
            for start in range(start_min, end):
                token: str = word[start:end]
                if token not in self.prob:
                    continue
                cand: float = best_score[start] - math.log(self.prob[token])
                if cand < best_score[end]:
                    best_score[end] = cand
                    best_path[end] = start

        if best_path[n] == -1:
            # Fallback: character-level segmentation (guaranteed by base chars).
            pieces: TokenSequence = list(word)
            loss: float = -sum(math.log(self.prob.get(ch, 1e-12)) for ch in pieces)
            return pieces, loss

        pieces: TokenSequence = []
        pos: int = n
        while pos > 0:
            start = best_path[pos]
            pieces.append(word[start:pos])
            pos = start
        pieces.reverse()
        return pieces, best_score[n]

    def _corpus_loss(self, word_freq: Counter[Tuple[str, ...]]) -> float:
        """Total negative log-likelihood under current token probabilities."""
        total_loss: float = 0.0
        for word_tuple, freq in word_freq.items():
            word: str = "".join(word_tuple)
            _, loss = self._viterbi_word(word)
            total_loss += loss * freq
        return total_loss

    def train(
        self,
        text: str,
        vocab_size: int = 300,
        shrink_ratio: float = 0.8,
        max_rounds: int = 8,
    ) -> "Unigram":
        """
        Train Unigram model by iterative re-estimation and pruning.

        @:param text: Preprocessed training corpus.
        @:param vocab_size: Final target vocabulary size.
        @:param shrink_ratio: Fraction of removable tokens kept per pruning round.
        @:parammax_rounds: Maximum prune/re-estimate cycles.
        """
        words: List[str] = self._extract_words(text)
        if not words:
            raise ValueError("Training text is empty after preprocessing.")

        self.base_chars = set(ch for word in words for ch in word)
        word_freq: Counter[Tuple[str, ...]] = Counter(tuple(word) for word in words)

        # Build a large seed vocabulary from substrings, then keep frequent ones.
        candidate_counts: Counter[str] = self._seed_vocab(words)
        kept_candidates: List[str] = [
            tok for tok, _ in candidate_counts.most_common(max(vocab_size * 4, 1000))
        ]

        # Ensure characters are always available.
        self.vocab = set(kept_candidates) | self.base_chars | {self.unk_token}
        init_usage: Counter[str] = Counter()
        for tok in self.vocab:
            init_usage[tok] = max(1, candidate_counts.get(tok, 0))
        self._normalize_probs(init_usage)

        for _ in range(max_rounds):
            # E-step style update: segment corpus with current probs, count usage.
            usage: Counter[str] = Counter()
            for word_tuple, freq in word_freq.items():
                word = "".join(word_tuple)
                pieces, _ = self._viterbi_word(word)
                for piece in pieces:
                    usage[piece] += freq

            # Keep all base chars and currently used tokens.
            for ch in self.base_chars:
                usage[ch] += 1
            usage[self.unk_token] += 1
            self._normalize_probs(usage)

            if len(self.prob) <= vocab_size:
                self.vocab = set(self.prob.keys())
                break

            # Loss-based pruning: remove tokens that hurt goal the least.
            removable: List[str] = [
                tok for tok in self.prob if tok not in self.base_chars and tok != self.unk_token
            ]
            if not removable:
                break

            current_loss: float = self._corpus_loss(word_freq)
            loss_increase: List[Tuple[float, str]] = []

            # For each token, temporarily drop it and measure objective degradation.
            for tok in removable:
                saved_prob = self.prob.pop(tok)
                test_loss = self._corpus_loss(word_freq)
                loss_increase.append((test_loss - current_loss, tok))
                self.prob[tok] = saved_prob

            loss_increase.sort(reverse=True)  # high increase => important token

            keep_removable: int = max(
                vocab_size - len(self.base_chars) - 1,
                int(len(removable) * shrink_ratio),
            )
            keep_tokens = {tok for _, tok in loss_increase[:keep_removable]}

            next_vocab = set(self.base_chars) | {self.unk_token} | keep_tokens
            self.prob = {tok: p for tok, p in self.prob.items() if tok in next_vocab}
            self.vocab = set(self.prob.keys())
            prob_total = sum(self.prob.values())
            if prob_total > 0:
                self.prob = {tok: p / prob_total for tok, p in self.prob.items()}

        self.vocab = set(self.prob.keys())
        return self

    def encode_word(self, word: str) -> TokenSequence:
        """Tokenize one word with best-probability segmentation."""
        pieces, _ = self._viterbi_word(word)
        if pieces:
            pieces[-1] = pieces[-1] + self.end_of_word
        return pieces

    def encode(self, text: str) -> TokenSequence:
        """Tokenize full text by whitespace words, then concatenate pieces."""
        tokens: TokenSequence = []
        for word in self._extract_words(text):
            tokens.extend(self.encode_word(word))
        return tokens

    def decode(self, tokens: Sequence[str]) -> str:
        """Rebuild words from `</w>` markers, then join words with spaces."""
        words: List[str] = []
        current: str = ""
        for token in tokens:
            if token.endswith(self.end_of_word):
                current += token[:-len(self.end_of_word)]
                words.append(current)
                current = ""
            else:
                current += token
        if current:
            words.append(current)
        return " ".join(words)


In [17]:
unigram = Unigram().train(clean_text, vocab_size=800)

print(f"Final vocab size: {len(unigram.vocab)}")

Final vocab size: 800


In [18]:
encoded = unigram.encode(sample_text)
decoded = unigram.decode(encoded)

print("\nSample text:", sample_text)
print("Encoded tokens:", encoded)
print("Decoded text:", decoded)


Sample text: Tokenizers are useful for efficient modeling.
Encoded tokens: ['Tokenizers</w>', 'are</w>', 'use', 'ful</w>', 'for</w>', 'eff', 'i', 'ci', 'ent</w>', 'model', 'ing.</w>']
Decoded text: Tokenizers are useful for efficient modeling.


### SentencePiece
Rather than splitting by frequency, a more "principled" approach is to define an objectvie function that captures what a good tokenization looks like.

It was one of the tokenizations supported in the **SentencePiece** tool, along with BPE.

Given a sequence $x_{1:L}$, a tokeniztion $T$ is a set of
$$
p(x_{1:L}) = \prod_{(i, j) \in T} p(x_{i:j})
$$

Example:
- training data: `ababc`
- $T = \{(1, 2), (3, 4), (5, 5)\} (\mathcal{V} = \{ab, c\})$
- Linkelihood: $p(x_{1:L}) = \frac{2}{3} \cdot \frac{2}{3} \cdot \frac{1}{3} = \frac{4}{9}$

**Algorithm**:
- start with a "reasonably big" seed vocabulary $\mathcal{V}$
- Repeat:
    - optimize $p(x)$ and $T$ using the EM algorithm
    - compute $loss(x)$ for each token $x \in \mathcal{V}$ capturing how much the likelihood would be reduced if $x$ were removed from $\mathcal{V}$
    - sort by loss and keep the top $80\%$ tokens in $\mathcal{V}$, and remove the rest.

In [20]:
from collections import Counter
from typing import Dict, List, Sequence, Tuple

SPToken = str
SPTokenSequence = List[SPToken]

class SentencePiece:
    """
    Simplified SentencePiece-Unigram tokenizer trained with EM.

    This implementation focuses on the core Unigram LM idea:
    - E-step: compute expected token counts via forward-backward.
    - M-step: update token probabilities from expected counts.
    - Periodically prune low-importance tokens.
    """

    def __init__(
        self,
        max_piece_length: int = 12,
        unk_token: str = "<unk>",
        end_of_word: str = "</w>",
    ) -> None:
        self.max_piece_length: int = max_piece_length
        self.unk_token: str = unk_token
        self.end_of_word: str = end_of_word

        # Model parameters: token -> probability and log-probability.
        self.prob: Dict[SPToken, float] = {}
        self.log_prob: Dict[SPToken, float] = {}
        self.vocab: set[str] = set()
        self.base_chars: set[str] = set()

    @staticmethod
    def _logsumexp(values: Sequence[float]) -> float:
        """Stable log(sum(exp(values)))."""
        if not values:
            return float("-inf")
        m = max(values)
        if m == float("-inf"):
            return m
        return m + math.log(sum(math.exp(v - m) for v in values))

    @staticmethod
    def _extract_words(text: str) -> List[str]:
        """Simple whitespace word extraction for training/encoding."""
        return re.findall(r"\S+", text)

    def _seed_vocab(self, words: Sequence[str], seed_size: int) -> set[str]:
        """Create initial candidate vocabulary from frequent substrings."""
        counts: Counter[str] = Counter()
        for word in words:
            n = len(word)
            for i in range(n):
                for j in range(i + 1, min(n, i + self.max_piece_length) + 1):
                    counts[word[i:j]] += 1

        seed_tokens = {tok for tok, _ in counts.most_common(seed_size)}
        return seed_tokens | self.base_chars | {self.unk_token}

    def _set_prob_from_counts(self, counts: Counter[str], smoothing: float = 1e-8) -> None:
        """M-step: normalize expected counts into token probabilities."""
        total = float(sum(counts.values())) + smoothing * max(1, len(counts))
        self.prob = {tok: (cnt + smoothing) / total for tok, cnt in counts.items() if cnt >= 0}
        self.log_prob = {tok: math.log(p) for tok, p in self.prob.items()}
        self.vocab = set(self.prob.keys())

    def _forward_backward(self, word: str) -> Tuple[List[float], List[float], float]:
        """
        Forward-backward on token lattice for one word.
        Returns alpha, beta, and log Z(word).
        """
        n = len(word)
        alpha = [float("-inf")] * (n + 1)
        beta = [float("-inf")] * (n + 1)
        alpha[0] = 0.0

        # Forward pass: alpha[pos] = log prob of all segmentations up to pos.
        for i in range(n):
            if alpha[i] == float("-inf"):
                continue
            for j in range(i + 1, min(n, i + self.max_piece_length) + 1):
                tok = word[i:j]
                logp = self.log_prob.get(tok)
                if logp is None:
                    continue
                alpha[j] = self._logsumexp([alpha[j], alpha[i] + logp])

        # Backward pass: beta[pos] = log prob of all segmentations from pos to end.
        beta[n] = 0.0
        for i in range(n - 1, -1, -1):
            candidates: List[float] = []
            for j in range(i + 1, min(n, i + self.max_piece_length) + 1):
                tok = word[i:j]
                logp = self.log_prob.get(tok)
                if logp is None or beta[j] == float("-inf"):
                    continue
                candidates.append(logp + beta[j])
            beta[i] = self._logsumexp(candidates)

        return alpha, beta, alpha[n]

    def _expected_counts(self, word: str, freq: int) -> Tuple[Counter[str], float]:
        """E-step contribution of one word: expected token counts and log-likelihood."""
        alpha, beta, logz = self._forward_backward(word)
        if logz == float("-inf"):
            # Should not happen for training words (base chars are kept), but keep safe fallback.
            fallback = Counter({ch: freq for ch in word})
            return fallback, float("-inf")

        n = len(word)
        expected: Counter[str] = Counter()

        # Posterior of token t at span (i, j):
        # exp(alpha[i] + log p(t) + beta[j] - log Z(word))
        for i in range(n):
            if alpha[i] == float("-inf"):
                continue
            for j in range(i + 1, min(n, i + self.max_piece_length) + 1):
                tok = word[i:j]
                logp = self.log_prob.get(tok)
                if logp is None or beta[j] == float("-inf"):
                    continue
                posterior = math.exp(alpha[i] + logp + beta[j] - logz)
                expected[tok] += posterior * freq

        return expected, logz * freq

    def _prune_vocab(
        self,
        expected_total: Counter[str],
        vocab_size: int,
        keep_ratio: float,
    ) -> None:
        """Drop low-count non-base tokens while preserving segmentability."""
        fixed = set(self.base_chars) | {self.unk_token}
        removable = [tok for tok in self.vocab if tok not in fixed]
        if not removable:
            return

        target_removable = max(0, vocab_size - len(fixed))
        keep_by_ratio = max(1, int(len(removable) * keep_ratio))
        keep_count = max(target_removable, keep_by_ratio)

        ranked = sorted(removable, key=lambda t: expected_total.get(t, 0.0), reverse=True)
        keep_removable = set(ranked[:keep_count])

        self.vocab = fixed | keep_removable
        filtered = Counter({tok: expected_total.get(tok, 0.0) for tok in self.vocab})
        for ch in self.base_chars:
            filtered[ch] += 1e-3
        filtered[self.unk_token] += 1e-3
        self._set_prob_from_counts(filtered)

    def train(
        self,
        text: str,
        vocab_size: int = 300,
        num_em_iters: int = 8,
        prune_every: int = 2,
        keep_ratio: float = 0.8,
    ) -> "SentencePiece":
        """
        Train with EM on a Unigram language model over subword pieces.
        """
        words = self._extract_words(text)
        if not words:
            raise ValueError("Training text is empty after preprocessing.")

        word_freq: Counter[str] = Counter(words)
        self.base_chars = set(ch for word in words for ch in word)

        seed_size = max(vocab_size * 6, 1200)
        self.vocab = self._seed_vocab(words, seed_size=seed_size)

        # Initialize probabilities from the raw frequency of candidate pieces.
        init_counts: Counter[str] = Counter()
        for word, freq in word_freq.items():
            n = len(word)
            for i in range(n):
                for j in range(i + 1, min(n, i + self.max_piece_length) + 1):
                    tok = word[i:j]
                    if tok in self.vocab:
                        init_counts[tok] += freq
        for ch in self.base_chars:
            init_counts[ch] += 1
        init_counts[self.unk_token] += 1
        self._set_prob_from_counts(init_counts)

        for em_iter in range(1, num_em_iters + 1):
            expected_total: Counter[str] = Counter()
            total_log_likelihood = 0.0

            # E-step: accumulate expected token counts under current model.
            for word, freq in word_freq.items():
                expected, ll = self._expected_counts(word, freq)
                expected_total.update(expected)
                if ll != float("-inf"):
                    total_log_likelihood += ll

            # Keep base characters valid and avoid zero-probability collapse.
            for ch in self.base_chars:
                expected_total[ch] += 1e-3
            expected_total[self.unk_token] += 1e-3

            # M-step: update parameters from expected counts.
            self._set_prob_from_counts(expected_total)

            # Prune every few rounds to shrink toward target vocabulary.
            if em_iter % prune_every == 0 and len(self.vocab) > vocab_size:
                self._prune_vocab(expected_total, vocab_size=vocab_size, keep_ratio=keep_ratio)

        # Final trim to exactly target size (when possible).
        if len(self.vocab) > vocab_size:
            self._prune_vocab(Counter(self.prob), vocab_size=vocab_size, keep_ratio=0.0)

        return self

    def encode_word(self, word: str) -> SPTokenSequence:
        """Viterbi decode: best-probability segmentation for one word."""
        n = len(word)
        best_score = [float("-inf")] * (n + 1)
        best_prev = [-1] * (n + 1)

        best_score[0] = 0.0
        for i in range(n):
            if best_score[i] == float("-inf"):
                continue
            for j in range(i + 1, min(n, i + self.max_piece_length) + 1):
                tok = word[i:j]
                logp = self.log_prob.get(tok)
                if logp is None:
                    continue
                cand = best_score[i] + logp
                if cand > best_score[j]:
                    best_score[j] = cand
                    best_prev[j] = i

        if best_prev[n] == -1:
            # Fallback for unseen chars: tokenize to chars/unk.
            pieces = [ch if ch in self.vocab else self.unk_token for ch in word]
            if pieces:
                pieces[-1] = pieces[-1] + self.end_of_word
            return pieces

        pieces: SPTokenSequence = []
        pos = n
        while pos > 0:
            prev = best_prev[pos]
            pieces.append(word[prev:pos])
            pos = prev
        pieces.reverse()
        pieces[-1] = pieces[-1] + self.end_of_word
        return pieces

    def encode(self, text: str) -> SPTokenSequence:
        tokens: SPTokenSequence = []
        for word in self._extract_words(text):
            tokens.extend(self.encode_word(word))
        return tokens

    def decode(self, tokens: Sequence[str]) -> str:
        words: List[str] = []
        current = ""
        for token in tokens:
            if token.endswith(self.end_of_word):
                current += token[:-len(self.end_of_word)]
                words.append(current)
                current = ""
            else:
                current += token
        if current:
            words.append(current)
        return " ".join(words)


In [23]:
spm = SentencePiece(max_piece_length=12).train(
    clean_text,
    vocab_size=800,
    num_em_iters=8,
    prune_every=2,
    keep_ratio=0.8,
)

print(f"SentencePiece vocab size: {len(spm.vocab)}")
print("Top 20 pieces by probability:")
for tok, p in sorted(spm.prob.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{tok!r}: {p:.6f}")

SentencePiece vocab size: 800
Top 20 pieces by probability:
'.': 0.017796
's': 0.016888
'and': 0.016120
'e': 0.009394
'"': 0.009338
'a': 0.008985
't': 0.008552
'in': 0.007893
'tokenizer': 0.007845
'r': 0.007434
',': 0.007277
're': 0.007248
'-': 0.007126
'1': 0.006733
'E': 0.006333
'A': 0.006331
'ing': 0.005919
'b': 0.005677
'n': 0.005611
'P': 0.005409


In [24]:
sp_encoded = spm.encode(sample_text)
sp_decoded = spm.decode(sp_encoded)

print("\nSample text:", sample_text)
print("SentencePiece encoded:", sp_encoded)
print("SentencePiece decoded:", sp_decoded)


Sample text: Tokenizers are useful for efficient modeling.
SentencePiece encoded: ['Tokenizers</w>', 'are</w>', 'useful</w>', 'for</w>', 'eff', 'i', 'ci', 'en', 't</w>', 'model', 'ing', '.</w>']
SentencePiece decoded: Tokenizers are useful for efficient modeling.
